<!-- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building [Synapsa](https://synapsa.realai.eu), an AI-native
learning platform.

© 2026 RealAI · free to learn from, share and adapt, not to sell ([CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)).
The notice at the end of this notebook says what you may and may not do.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T01-L02-leakage-clinic/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/lessons/T01-L02-leakage-clinic/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=lessons/T01-L02-leakage-clinic/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy, which Colab, Kaggle, Binder and
Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v4 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/lessons/T01-L02-leakage-clinic/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# T01-L02 · How models lie: a leakage and validation clinic

**You will build:** a clinic for evaluation protocols: a seeded book of customer records with five
leaks planted in it, a measurement of what each leak is worth in cross-validated AUC, and the split,
feature and pipeline checks that catch all five.

**Time:** ~50 minutes · **Runs on:** a laptop CPU, no GPU, no download ·
**Prerequisites:** T00-L01-the-8gb-track (the numpy toolchain this course runs on) and
T01-L01-statistics-by-simulation (means, standard errors, and why one run proves little).

A cross-validated AUC is a promise: "on customers like these, this model will rank the ones who
ask to cancel above the ones who do not this well". Five different mistakes let a model keep that promise on
paper and break it in production, and each hides in a different place: in a column, in a
preprocessing step, in the rows, in the customers, or in the calendar. Real data never tells you
which mistake you made. Here the book is generated, so every leak is planted, its cost can be
measured, and there is a ceiling that no honest model can pass.

By the end you will be able to:

1. Measure how much each of five planted leaks (target, preprocessing, duplicate, group and
   temporal) inflates cross-validated AUC, against an honest protocol and an oracle ceiling.
2. Implement a split audit that counts the groups, test rows and repeated rows that cross the
   train/test boundary of one fold.
3. Implement a feature audit (a point-in-time lateness check and a too-good-to-be-true scan) and
   a fold-isolation test that detects a pipeline using test labels or other test rows.
4. Implement a protocol diagnosis that combines the checks, show the checks fall silent as the leaks
   are fixed in one fixed order, and find the folds under which the duplicate check is silent while
   the leak is still there.
5. Implement the average step of each leak over every order of fixing, and use it to show that a
   leak's measured cost depends on what else is still broken.

<!-- COMMONS BEFORE v1 · generated from meta.yaml by tools/notebooks.py · do not edit by hand -->
## Before you start

**Lessons to do first:** `T00-L01`, `T01-L01`

**This lesson assumes you can already:**

- write Python functions and loops, and index a numpy array with an array of row numbers or with a boolean mask
- say what AUC measures, as the chance that a randomly chosen positive row is scored above a randomly chosen negative one
- describe k-fold cross-validation: train on all folds but one, score on the held-out fold, and repeat for each fold
- read a mean with its standard error and say whether a gap between two means is larger than its noise (T01-L01 builds this)

**New to some of that? These are good places to learn it** (free to read online first, then paid books). You do not need any of them if the list above already reads as familiar.

- [Common pitfalls and recommended practices, section 12.2 Data leakage](https://scikit-learn.org/stable/common_pitfalls.html) · scikit-learn developers · *free* — defines data leakage and shows preprocessing and feature selection done before the split, the failure the fold-isolation test in exercise 3 is built to catch
- [Cross-validation: evaluating estimator performance (iterators for grouped data and for time series)](https://scikit-learn.org/stable/modules/cross_validation.html) · scikit-learn developers · *free* — the group k-fold and time-series split sections describe the two split families whose boundary exercise 1 audits
- [Leakage and the reproducibility crisis in machine-learning-based science](https://pmc.ncbi.nlm.nih.gov/articles/PMC10499856/) · Sayash Kapoor and Arvind Narayanan (Patterns, 2023, open access) · *free* — a survey of leakage in published research with a taxonomy that contains this lesson's five leaks (pre-processing, duplicates, illegitimate features, temporal leakage, non-independence)
- [Classification: ROC and AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc) · Google for Developers, Machine Learning Crash Course · *free* — reads AUC as the chance that a randomly chosen positive example is ranked above a randomly chosen negative one, the reading behind exercise 2's too-good scan and every AUC this lesson prints
- [Indexing on ndarrays](https://numpy.org/doc/stable/user/basics.indexing.html) · The NumPy project · *free* — the sections on integer array indexing and boolean array indexing, the two ways every exercise picks out the rows of a fold

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import itertools
import math
import os
import signal
import sys
import threading
import time
import traceback
from typing import Callable, Mapping, NamedTuple, Sequence

import numpy as np

print("python", sys.version.split()[0], "· numpy", np.__version__, "· platform", sys.platform)

SEED = 20261014              # this lesson's one seed: every book derives from it, so re-running
                             # prints the same numbers anywhere
N_BOOKS = 12                 # how many independently generated books every measurement averages over
HORIZON_DAYS = 30            # the label looks this far ahead, so a training row is only usable for a
                             # test row at least this many days later
ISOLATION_TOL = 1e-9         # a score that moves by more than this when it should not has leaked
AUC_BOUND = 0.9              # the "too good to be true" bound for one column's AUC on its own
CV_FOLDS = 5                 # folds for the two random-split protocols

_FAILED_CHECKS: list[str] = []
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run

# The exercises, in the order you meet them, and the functions each one asks you to write. The
# progress board at the foot of the notebook is built from this, and a cell that is waiting on an
# unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("split_leaks",),
    "exercise 2": ("feature_audit",),
    "exercise 3": ("fold_isolation",),
    "exercise 4": ("diagnose",),
    "exercise 5": ("average_steps",),
}


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (fold_isolation)"; several -> "exercises 2, 3 and 4"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


# A check that never returns would freeze the notebook (and the grader, which imports it). Every
# check below runs under a time limit far above what a correct answer needs, and so does every
# single call a demo makes into your code; a whole demo gets a much longer limit, because slow
# but correct code is allowed to be slow.
_CHECK_SECONDS = 2
_CALL_SECONDS = 3
_DEMO_SECONDS = 90

# A time limit for every check that calls the student's code: a loop that never ends must fail
# its check, not freeze the notebook or the grader (which imports the notebook). Copy this block
# unchanged; change only the message. Why it is this careful:
#  * Whoever runs the notebook may already have an alarm running (tools/grade.py stops every
#    rubric row after 45 s; a harness may have one whose default action ends the process). This
#    guard saves it, lets whichever deadline comes first win (handing the signal to the outer
#    handler when it is the outer's turn) and on the way out re-arms it with the time it had left
#    and its interval. Switching it off instead would let the NEXT hang run for ever.
#  * Its own alarm is one-shot, re-armed by hand every 0.25 s once the time is up (in case a bare
#    `except:` swallows the first). A repeating alarm kept firing while a runaway loop's huge list
#    was being torn down; a repeat that arrived after the default handler was back ended the
#    process (exit -14, a dead Jupyter kernel), and one that arrived inside contextlib's exit
#    escaped as a raw timeout and left this handler installed.
#  * It never raises inside its own set-up, exit or clean-up, and on the way out it takes any
#    alarm of its own that has gone off but not yet been handled (with SIGALRM blocked) before
#    the old handler is put back.
# tools/verify_all.py fails a lesson whose guard does less.
class _OutOfTime(BaseException):
    """Raised inside the guarded block when its time is up. A BaseException, so `except
    Exception:` in the code under test cannot swallow it; it repeats every 0.25 s in case a bare
    `except:` does. `_time_limit` turns it into an AssertionError once it has left that code."""


def _resident() -> float:
    """This process's resident memory now, in MiB (Linux: /proc; macOS: libproc); its peak so far elsewhere."""
    try:
        if sys.platform.startswith("linux"):
            with open("/proc/self/statm") as f:
                return int(f.read().split()[1]) * os.sysconf("SC_PAGE_SIZE") / 2**20
        if sys.platform == "darwin":
            import ctypes, struct
            info = ctypes.create_string_buffer(96)           # struct proc_taskinfo
            if ctypes.CDLL("/usr/lib/libproc.dylib").proc_pidinfo(os.getpid(), 4, 0, info, 96) == 96:
                return struct.unpack_from("QQ", info.raw)[1] / 2**20     # pti_resident_size
    except Exception:
        pass
    import resource
    return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (2**20 if sys.platform == "darwin" else 2**10)


@contextlib.contextmanager
def _time_limit(seconds: float, what: str, memory_mib: float = 2048):
    """Fail the block with an AssertionError naming `what` if it runs longer than `seconds`.
    Uses the operating system's alarm (Linux and macOS, main thread); a no-op elsewhere."""
    if not (all(hasattr(signal, name) for name in ("setitimer", "SIGALRM", "pthread_sigmask", "sigpending", "sigwait"))
            and threading.current_thread() is threading.main_thread()):
        yield
        return
    if getattr(_time_limit, "stopped", False):           # a stopped block's frames keep what they built in a
        _time_limit.stopped = False                      # reference cycle until now: free it, so it is not counted
        import gc
        gc.collect()                                     # as the growth of this block
    message = (f"{what} was still running after {seconds:g} s: a loop whose stopping condition "
               "is never met runs for ever; give every loop a cap")
    started = time.monotonic()
    used_mib = None
    if memory_mib:
        used_mib = _resident
        base_mib = used_mib()                            # resident memory now; only growth beyond it counts
    previous = signal.getsignal(signal.SIGALRM)
    outer_left, outer_every = signal.getitimer(signal.ITIMER_REAL)   # an alarm already running
    # One alarm at a time, never a repeating one. next: when ours is due; outer: when the alarm we
    # found is due (every: its interval); unseen: an alarm is armed and has not reached _fire yet.
    state = {"next": started + seconds, "outer": started + outer_left if outer_left > 0 else None,
             "every": outer_every, "unseen": False, "fired": False, "done": False, "message": message, "count": 0, "budget": 0,
             "traced": False, "tracer": None, "tool": None, "at": None, "thread": threading.get_ident(),
             "me": sys._getframe(), "owner": None}
    owner = state["me"].f_back
    while owner is not None and owner.f_globals is vars(contextlib):    # contextlib's __enter__
        owner = owner.f_back
    state["owner"], state["at"] = owner, owner.f_lineno   # the frame running the `with`, and its line

    def _arm(now, soonest=0.001):                        # one shot, at whichever deadline is next
        due = [t for t in (state["next"], state["outer"]) if t is not None]
        if used_mib and state["next"] is not None and not state["fired"]:
            due.append(now + 0.1)                        # and look at the memory every 0.1 s
        if due:
            signal.setitimer(signal.ITIMER_REAL, max(min(due) - now, soonest))
            state["unseen"] = True

    def _in_guard(frame):                                # is the alarm landing in a guard's own set-up, exit
        while frame is not None and frame is not state["owner"]:     # or clean-up (this one's or another's:
            code = frame.f_code                          # guards nest), or in contextlib's __exit__ under
            if (frame is state["me"] or code in _own_code or code.co_name in _GUARDS   # a `with`?
                    or frame.f_globals.get("__name__") == "contextlib"):
                return True
            frame = frame.f_back
        return False

    def _fire(signum, frame):
        state["unseen"] = False
        if state["done"]:
            return                                       # a late alarm of ours: already handled
        if _in_guard(frame):                             # never raise inside the guard: it is
            _arm(time.monotonic(), 0.01)                 # finishing; look again shortly
            return
        now = time.monotonic()
        if state["outer"] is not None and now >= state["outer"] - 0.005:
            every = state["every"]                       # the outer alarm is due: act as it would
            state["outer"] = state["outer"] + every * (1 + (now - state["outer"]) // every) if every else None
            try:
                if callable(previous):
                    previous(signum, frame)
                elif previous != signal.SIG_IGN:         # its default action ends the process
                    signal.signal(signal.SIGALRM, signal.SIG_DFL)
                    os.kill(os.getpid(), signal.SIGALRM)
            except BaseException:
                state["next"] = None                     # the outer one is stopping the block
                raise
            finally:
                left, every = signal.getitimer(signal.ITIMER_REAL)
                if left > 0:                             # the outer handler armed its next alarm
                    state["outer"], state["every"] = time.monotonic() + left, every
                _arm(time.monotonic())
            now = time.monotonic()
        grown = used_mib() - base_mib if used_mib and not state["fired"] else 0
        if grown > memory_mib and state["next"] is not None:       # out of memory before out of time
            state["message"] = (f"{what} was using over {memory_mib:g} MiB more memory than when it started, "
                                f"after {now - started:.1f} s: a loop that keeps adding to a list or a string "
                                "never stops; give every loop a cap")
            state["next"] = now
        if state["next"] is not None and now >= state["next"] - 0.005:
            swallowed = state["fired"]                   # a second stop: the first did not end the block
            state["fired"] = True
            state["count"] += 1
            state["next"] = now + 0.25                   # again, in case a bare `except:` swallows it
            _arm(now)
            if swallowed:
                state["budget"] = state["count"] - 1     # the tracer raises this many times, then leaves clean-up alone
                _trace_from(frame)
            raise _OutOfTime(state["message"], state)
        _arm(now)

    def _trace(frame, event, arg):                       # raises the stop at the next line of code that
        if state["done"] or _in_guard(frame):            # runs: inside the handler that swallowed the last
            return None                                  # (the interpreter then unsets it; the next alarm
        if event == "line":                              # sets it again)
            raise _OutOfTime(state["message"], state)
        return _trace

    def _line(code, line):                               # Python 3.12+: sys.monitoring, which is not unset
        if state["done"] or state["budget"] <= 0 or threading.get_ident() != state["thread"]:   # not unset when a
            return                                       # callback raises: the next line the code runs raises too, handlers
        frame = sys._getframe(1)                         # included (this thread only), but only `budget` times:
        if _in_guard(frame) or (frame is state["owner"] and line == state["at"]):   # clean-up that runs once the stop has
            return                                       # left the swallowing loops must not be interrupted again
        state["budget"] -= 1                             # (not the `with` line: its exit runs the guard)
        raise _OutOfTime(state["message"], state)

    def _trace_from(frame):                              # trace the running frames, and the ones called next
        if not state["traced"]:
            state["traced"] = True
            mon = getattr(sys, "monitoring", None)
            for tool in (5, 4, 3, 2, 1, 0) if mon is not None else ():
                try:
                    mon.use_tool_id(tool, "commons-time-limit")
                except ValueError:                       # a debugger or a profiler has this one
                    continue
                state["tool"] = tool
                mon.register_callback(tool, mon.events.LINE, _line)
                mon.set_events(tool, mon.events.LINE)
                return
            state["tracer"] = sys.gettrace()
            sys.settrace(_trace)
        while frame is not None and frame is not state["owner"]:
            frame.f_trace = _trace
            frame = frame.f_back

    _GUARDS = ("_time_limit", "_deadline", "_resident", "_time_limit", "_fire", "_arm", "_trace", "_trace_from", "_line", "_in_guard")
    _own_code = (_fire.__code__, _arm.__code__, _trace.__code__, _trace_from.__code__, _line.__code__)

    signal.signal(signal.SIGALRM, _fire)
    _arm(time.monotonic())
    try:
        try:
            yield
        finally:
            state["done"] = True                         # from here on an alarm of ours only counts
            if state["traced"]:
                state["traced"] = False
                if state["tool"] is not None:
                    sys.monitoring.set_events(state["tool"], 0)
                    sys.monitoring.register_callback(state["tool"], sys.monitoring.events.LINE, None)
                    sys.monitoring.free_tool_id(state["tool"])
                else:
                    sys.settrace(state["tracer"])        # the tracer that was there before, if any
            mask = signal.pthread_sigmask(signal.SIG_BLOCK, {signal.SIGALRM})
            try:
                if signal.setitimer(signal.ITIMER_REAL, 0)[0] == 0 and state["unseen"]:
                    # Ours went off and has not reached _fire: it is pending here, or on its way to
                    # another thread. Take it before the old handler (by default, one that ends the
                    # process) is back.
                    give_up = time.monotonic() + 0.5
                    while state["unseen"] and time.monotonic() < give_up:
                        if signal.SIGALRM in signal.sigpending():
                            signal.sigwait({signal.SIGALRM})
                            state["unseen"] = False
                        else:
                            time.sleep(0.001)            # another thread has it: _fire runs next
                signal.signal(signal.SIGALRM, signal.SIG_DFL if previous is None else previous)
                outer = state["outer"]                   # re-arm the alarm we found, last
                signal.setitimer(signal.ITIMER_REAL, *((max(outer - time.monotonic(), 0.001), state["every"])
                                                       if outer is not None else (0, 0)))
            finally:
                signal.pthread_sigmask(signal.SIG_SETMASK, mask)
                state["me"] = state["owner"] = None
    except _OutOfTime as exc:
        if exc.args[-1] is not state:                    # an enclosing guard's: not ours to report
            raise
        _time_limit.stopped = True
        raise AssertionError(state["message"]) from None
    if state["fired"]:                                   # the block caught our alarm and carried on
        _time_limit.stopped = True
        raise AssertionError(state["message"])


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message, which names the likely mistake, and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo (and the exercise 4 check, which
    calls your first three functions) names the exercises it `needs`: until each has passed its
    check, it says which one it is waiting for and skips, so a fault in an earlier exercise is
    never reported as a fault in a later one.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        with _time_limit(_CHECK_SECONDS if label in _EXERCISES else _DEMO_SECONDS, label):
            check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"


def _call(what: str, fn: Callable, *args, **kwargs):
    """Call your function from a demo under its own short time limit, so one runaway loop in an
    exercise costs seconds and a failed demo line, not the notebook."""
    with _time_limit(_CALL_SECONDS, what):
        return fn(*args, **kwargs)


# Demos (the tables below the exercises) run when this file is the program being run, that is in a
# notebook or as `python lesson.py`; the autograder imports the file and only needs the checks.
_DEMOS = __name__ == "__main__"

## 1. A subscription book with five leaks planted

A subscription service keeps one row per customer per month. The label is whether the customer
**asks to cancel within `HORIZON_DAYS` days of the row's prediction date** (calls the cancellation
line). It is a request, not an exit: the retention team can talk a customer round, so the same
customer can ask again in a later month and keeps having rows.

**How the model is used.** It is trained on the history up to some date and then, each month, ranks
the customers who **joined after that date** by how likely they are to ask. That one sentence
decides what an honest split must imitate: test rows that are later in time and belong to customers
the model has never seen. (A model that re-scored the customers it was trained on would face a
different question, and the group leak below would not be a leak.)

Each row has three usage signals and three columns a model can use as shortcuts. The five leaks
are planted on purpose:

* **target**: `retention_offer` is set by the retention team *after* a customer asks to cancel.
  It is in the row, so it looks like any other feature, but it was not known at prediction time.
* **preprocessing**: `postcode` and `month` are categories, and the pipeline turns each into a
  number by averaging the label inside the category (a *target encoding*). Done once on all
  rows, before splitting, the test rows' own labels end up inside their own features.
* **duplicate**: some accounts were registered twice. A customer's whole history appears again
  under a new customer id, row for row.
* **group**: a customer appears in several months, and each customer has a stable tendency to ask
  to cancel. Every customer has a `postcode` that no other customer has, so it works as a customer
  id and a model can memorise it. That is a leak here because
  the deployed model meets customers it has not seen. For a model that re-scores its own customers a
  customer's earlier rows are fair evidence, and the source in `claims.yaml` says nonindependence is
  leakage "unless" the claim is about a distribution with the same dependence structure.
* **temporal**: each calendar month has its own shock to how often customers ask to cancel, which
  only the rows of that month reveal. `month` lets a model learn the shock from the other rows of
  the same month, which a random split puts in the training fold (along with rows from later months).

Everything is drawn from `SEED`, so every number below is the same on every machine. The customer
and month effects (`CUSTOMER_SD`, `MONTH_SD`) are set on purpose to be as strong as the usage
signals: at weaker settings a trial run left the group and duplicate costs lost in the noise.
Real books will differ, so read every size printed here as what this book shows, not as typical.
The cell builds `N_BOOKS` independent books; each measurement in this lesson averages over them.
Read the cell below; nothing in it is graded.

In [ ]:
N_CUSTOMERS = 360                 # customers per book, before duplicates are added
N_MONTHS = 24                     # calendar months in a book
MIN_ROWS, MAX_ROWS = 3, 6         # each customer has 3 to 6 consecutive monthly rows
DAYS_PER_MONTH = 30
DUP_SHARE = 0.12                  # share of customers whose whole history is registered twice
COEF = (0.9, 0.6, 0.4)            # true weights of the three usage signals in the logit of asking to cancel
INTERCEPT = -1.4
CUSTOMER_SD = 1.5                 # spread of each customer's stable tendency to ask to cancel
MONTH_SD = 1.0                    # spread of each calendar month's shock to how often customers ask
OFFER_IF_ASKED = 0.75             # chance retention_offer is set in a row whose customer then asks to cancel
OFFER_IF_NOT_ASKED = 0.02         # ... and in a row whose customer does not

NAMES = ("usage_drop", "support_tickets", "late_payments", "retention_offer", "postcode", "month")
LATE_FEATURE = "retention_offer"
CATEGORICAL = (4, 5)              # columns of X that are category ids, not measurements
LEAKS = ("target", "preprocessing", "duplicate", "group", "temporal")   # the order of the fixes


class Book(NamedTuple):
    """One generated book. Row i of every array describes the same customer-month."""
    names: tuple                  # the column names of X and written_at
    X: np.ndarray                 # (rows, columns) floats; the category columns hold integer ids
    y: np.ndarray                 # 1 if the customer asked to cancel within HORIZON_DAYS of predicted_at
    customer: np.ndarray          # customer id: the group a row belongs to
    predicted_at: np.ndarray      # the day (an integer) the model would have scored this row
    written_at: np.ndarray        # (rows, columns) the day each value was written to the record
    keys: list                    # one hashable per row: its feature values, labels excluded


def make_book(book_seed: int) -> Book:
    """Generate one book with every leak planted. Given.

    Customer i has `n_rows[i]` consecutive monthly rows starting at `signup[i]`. The logit of asking
    to cancel is the three usage signals (weights COEF), plus the customer's stable effect, plus the
    calendar month's shock. `postcode` is a number that is unique to each customer, so it works as a
    customer id. A share DUP_SHARE of customers is then copied: every row again, same values, under a
    fresh customer id.
    """
    rng = np.random.default_rng([SEED, book_seed])
    signup = rng.integers(0, N_MONTHS - MAX_ROWS + 1, N_CUSTOMERS)
    n_rows = rng.integers(MIN_ROWS, MAX_ROWS + 1, N_CUSTOMERS)
    postcode_of = rng.permutation(N_CUSTOMERS)
    customer_effect = CUSTOMER_SD * rng.standard_normal(N_CUSTOMERS)
    month_effect = MONTH_SD * rng.standard_normal(N_MONTHS)

    customer = np.repeat(np.arange(N_CUSTOMERS), n_rows)
    first_month = np.repeat(signup, n_rows)
    month = first_month + np.concatenate([np.arange(k) for k in n_rows])
    n = len(customer)
    usage = rng.standard_normal((n, 3))
    logit = (INTERCEPT + usage[:, 0] * COEF[0] + usage[:, 1] * COEF[1] + usage[:, 2] * COEF[2]
             + customer_effect[customer] + month_effect[month])
    y = (rng.random(n) < 1.0 / (1.0 + np.exp(-logit))).astype(np.int64)
    offered = np.where(y == 1, rng.random(n) < OFFER_IF_ASKED, rng.random(n) < OFFER_IF_NOT_ASKED)

    predicted_at = month * DAYS_PER_MONTH
    X = np.column_stack([usage, offered.astype(float), postcode_of[customer].astype(float),
                         month.astype(float)])
    written_at = np.empty((n, 6), dtype=np.int64)
    written_at[:, :3] = predicted_at[:, None] - rng.integers(1, 8, (n, 3))   # measured the week before
    written_at[:, 3] = np.where(offered, predicted_at + 3 + rng.integers(0, 18, n), predicted_at - 1)
    written_at[:, 4] = (first_month * DAYS_PER_MONTH - 7)                    # written at signup
    written_at[:, 5] = predicted_at                                          # the calendar, at the time

    copied = np.sort(rng.permutation(N_CUSTOMERS)[:int(round(DUP_SHARE * N_CUSTOMERS))])
    rows = np.flatnonzero(np.isin(customer, copied))
    new_id = N_CUSTOMERS + np.searchsorted(copied, customer[rows])
    X = np.vstack([X, X[rows]])
    y = np.concatenate([y, y[rows]])
    customer = np.concatenate([customer, new_id])
    predicted_at = np.concatenate([predicted_at, predicted_at[rows]])
    written_at = np.vstack([written_at, written_at[rows]])
    keys = [row.tobytes() for row in X]
    return Book(NAMES, X, y, customer, predicted_at, written_at, keys)


def take(book: Book, rows) -> Book:
    """The same book restricted to `rows` (an array of row numbers, in that order). Given."""
    rows = np.asarray(rows, dtype=np.int64)
    return Book(book.names, book.X[rows], book.y[rows], book.customer[rows],
                book.predicted_at[rows], book.written_at[rows], [book.keys[i] for i in rows])


def dedupe(book: Book) -> Book:
    """Keep the first row of every repeated key. Given."""
    seen, keep = set(), []
    for i, key in enumerate(book.keys):
        if key not in seen:
            seen.add(key)
            keep.append(i)
    return take(book, keep)


BOOKS = [make_book(i) for i in range(N_BOOKS)]

if _DEMOS:
    b = BOOKS[0]
    repeats = len(b.keys) - len(set(b.keys))
    late = int((b.written_at[:, NAMES.index(LATE_FEATURE)] > b.predicted_at).sum())
    ids, row_customer = np.unique(b.customer, return_inverse=True)
    asked = np.bincount(row_customer, weights=b.y, minlength=len(ids))
    print(f"book 0: {len(b.y)} rows, {len(ids)} customer ids, "
          f"{repeats} rows are repeats of another row, {b.y.mean():.1%} of rows are requests to cancel")
    print(f"{int((asked >= 2).sum())} of {len(ids)} customer ids ask to cancel in more than one month: the label is a "
          "request, which a customer can make again, not an exit")
    print(f"{N_BOOKS} books in all; columns: {', '.join(NAMES)}")
    print(f"{LATE_FEATURE} was written after the row's prediction date in {late} of {len(b.y)} rows; "
          f"month is stamped the same day as the prediction date in "
          f"{int((b.written_at[:, 5] == b.predicted_at).sum())} rows")
    for i in (0, 1, 2):
        vals = "  ".join(f"{n}={v:g}" for n, v in zip(NAMES, b.X[i]))
        print(f"  customer {b.customer[i]}, day {b.predicted_at[i]}: {vals}  asks_to_cancel={b.y[i]}")

## 2. The model, and a protocol you can break five ways

The model is a ridge logistic regression on the three usage signals, `retention_offer`, and the
two category columns turned into numbers by target encoding: each category becomes its smoothed
average label. An honest encoder learns those averages from the *training fold only*, and
encodes the training rows **out of fold** (cross-fitting), so no row ever sees its own label;
scikit-learn's `TargetEncoder` encodes training data this way for the same reason (`claims.yaml`).

An evaluation **protocol** is five yes/no choices, one per leak. `fixed` is the set of leaks a
protocol guards against:

| leak | fixing it means |
|---|---|
| `target` | leave out `retention_offer` |
| `preprocessing` | fit the scaler and the encoders on the training fold only |
| `duplicate` | drop repeated rows before splitting |
| `group` | split by customer, not by row |
| `temporal` | train only on rows at least `HORIZON_DAYS` before the test rows |

The **naive** protocol fixes nothing; the **honest** one fixes all five and so imitates the
deployment of section 1. The cell below is given (AUC from ranks, the model, the four ways of
splitting, and the protocol builder); read it, then run the one after it to see the gap between the
two. That cell also prints an **oracle ceiling** for each protocol: the AUC of the planted usage
weights themselves, on the same test rows that protocol uses. On average no model that uses only
the three usage signals can beat it on new customers, because the customer and month effects it
leaves out cannot be known in advance. Protocols differ in which rows they test on (a forward split
tests only later customers), which is why each is compared with its own ceiling and why their
scores are what each protocol would have told you, not one quantity measured twice.

In [ ]:
def auc_score(y_true, scores) -> float:
    """Area under the ROC curve by the rank (Mann-Whitney) identity, ties given the average rank.
    Given. Equals the chance that a random positive row outscores a random negative one, with
    ties counting half. Raises ValueError if only one class is present."""
    y = np.asarray(y_true)
    s = np.asarray(scores, dtype=float)
    n_pos = int(y.sum())
    n_neg = int(y.size - n_pos)
    if n_pos == 0 or n_neg == 0:
        raise ValueError("AUC is undefined when one class is absent")
    order = np.argsort(s, kind="mergesort")
    _, inverse, counts = np.unique(s[order], return_inverse=True, return_counts=True)
    ends = np.cumsum(counts)
    average_rank = (ends - counts + 1 + ends) / 2.0
    ranks = np.empty(s.size)
    ranks[order] = average_rank[inverse]
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


RIDGE = 1.0
NEWTON_STEPS = 12                 # a fixed count, not a tolerance: the same work on every machine
SMOOTHING = 5.0                   # pseudo-rows of the overall rate mixed into every category average
INNER_FOLDS = 5                   # cross-fitting folds for the training rows' encodings


def fit_logistic(A: np.ndarray, y: np.ndarray) -> np.ndarray:
    """Ridge logistic regression by Newton's method, intercept first. Given.
    Sums are written with einsum (plain loops, no BLAS), so they come out the same everywhere."""
    n, d = A.shape
    A1 = np.empty((n, d + 1))
    A1[:, 0] = 1.0
    A1[:, 1:] = A
    penalty = np.full(d + 1, RIDGE)
    penalty[0] = 0.0
    w = np.zeros(d + 1)
    for _ in range(NEWTON_STEPS):
        p = 1.0 / (1.0 + np.exp(-np.einsum("ij,j->i", A1, w)))
        gradient = np.einsum("ij,i->j", A1, p - y) + penalty * w
        hessian = np.einsum("ij,i,ik->jk", A1, p * (1.0 - p), A1) + np.diag(penalty)
        w = w - np.linalg.solve(hessian, gradient)
    return w


def linear_score(w: np.ndarray, A: np.ndarray) -> np.ndarray:
    """The model's score (log-odds) for the rows of A. Given."""
    return w[0] + np.einsum("ij,j->i", A, w[1:])


def _crossfit(cat: np.ndarray, y: np.ndarray, n_levels: int) -> np.ndarray:
    """Encode training rows out of fold: row i's category average never includes row i."""
    out = np.empty(len(cat))
    inner = np.arange(len(cat)) % INNER_FOLDS
    for f in range(INNER_FOLDS):
        held = inner == f
        s = np.bincount(cat[~held], weights=y[~held], minlength=n_levels)
        c = np.bincount(cat[~held], minlength=n_levels)
        out[held] = (s[cat[held]] + SMOOTHING * y[~held].mean()) / (c[cat[held]] + SMOOTHING)
    return out


def make_pipeline(used: Sequence[str], scaler_in_fold: bool = True, encoder_in_fold: bool = True):
    """Build a `fit_predict(X, y, train_idx, test_idx)` that returns one score per test row. Given.

    `used` names the columns of X it may read. With `encoder_in_fold` the category averages come
    from the training rows only and the training rows are encoded out of fold; without it they
    come from EVERY row's label (the test rows' own included), the classic mistake. With
    `scaler_in_fold` the standardising means and spreads come from the training rows only; without
    it, from the training and test rows together. Rows outside train and test are never scored.
    """
    cols = [j for j, name in enumerate(NAMES) if name in used]

    def fit_predict(X, y, train_idx, test_idx):
        tr = np.asarray(train_idx, dtype=np.int64)
        te = np.asarray(test_idx, dtype=np.int64)
        yf = np.asarray(y, dtype=float)
        label_rows = tr if encoder_in_fold else np.arange(len(yf))
        part_tr, part_te = [], []
        for j in cols:
            if j in CATEGORICAL:
                cat = X[:, j].astype(np.int64)
                n_levels = int(cat.max()) + 1
                s = np.bincount(cat[label_rows], weights=yf[label_rows], minlength=n_levels)
                c = np.bincount(cat[label_rows], minlength=n_levels)
                prior = yf[label_rows].mean()
                part_te.append((s[cat[te]] + SMOOTHING * prior) / (c[cat[te]] + SMOOTHING))
                if encoder_in_fold:
                    part_tr.append(_crossfit(cat[tr], yf[tr], n_levels))
                else:
                    part_tr.append((s[cat[tr]] + SMOOTHING * prior) / (c[cat[tr]] + SMOOTHING))
            else:
                part_tr.append(X[tr, j])
                part_te.append(X[te, j])
        A_tr, A_te = np.column_stack(part_tr), np.column_stack(part_te)
        reference = A_tr if scaler_in_fold else np.vstack([A_tr, A_te])
        mean, spread = reference.mean(axis=0), reference.std(axis=0) + 1e-12
        w = fit_logistic((A_tr - mean) / spread, yf[tr])
        return linear_score(w, (A_te - mean) / spread)

    return fit_predict


FORWARD_STARTS = tuple(m * DAYS_PER_MONTH for m in (9, 12, 15, 18))   # the four test windows
FORWARD_BLOCK = 3 * DAYS_PER_MONTH                                     # each is three months long


def random_folds(n_rows: int, k: int, seed: int) -> list:
    """k folds of randomly chosen rows. Given. Returns [(train_idx, test_idx), ...]."""
    order = np.random.default_rng([SEED, 1, seed]).permutation(n_rows)
    fold = np.empty(n_rows, dtype=np.int64)
    fold[order] = np.arange(n_rows) % k
    return [(np.flatnonzero(fold != f), np.flatnonzero(fold == f)) for f in range(k)]


def group_folds(groups: np.ndarray, k: int, seed: int) -> list:
    """k folds in which every group (customer) lies wholly in one fold. Given."""
    ids = np.unique(groups)
    order = np.random.default_rng([SEED, 2, seed]).permutation(len(ids))
    fold_of = np.empty(len(ids), dtype=np.int64)
    fold_of[order] = np.arange(len(ids)) % k
    fold = fold_of[np.searchsorted(ids, groups)]
    return [(np.flatnonzero(fold != f), np.flatnonzero(fold == f)) for f in range(k)]


def forward_folds(groups: np.ndarray, times: np.ndarray, by_group: bool) -> list:
    """Expanding-window folds in time. Given. A fold trains on the rows predicted at least
    HORIZON_DAYS before its window starts and tests on the window. With `by_group` the window holds
    the customers whose FIRST row falls inside it, with all their rows (so no customer is on both
    sides); without it, it holds every row that falls inside it (so a customer who straddles the
    boundary is on both sides)."""
    ids, inverse = np.unique(groups, return_inverse=True)
    first = np.full(len(ids), np.iinfo(np.int64).max)
    np.minimum.at(first, inverse, times)
    when = first[inverse] if by_group else times
    out = []
    for start in FORWARD_STARTS:
        out.append((np.flatnonzero(times <= start - HORIZON_DAYS),
                    np.flatnonzero((when >= start) & (when < start + FORWARD_BLOCK))))
    return out


def protocol_folds(book: Book, group_disjoint: bool, forward: bool, seed: int) -> list:
    """The four families of folds a protocol can use. Given."""
    if forward:
        return forward_folds(book.customer, book.predicted_at, by_group=group_disjoint)
    if group_disjoint:
        return group_folds(book.customer, CV_FOLDS, seed)
    return random_folds(len(book.y), CV_FOLDS, seed)


ALL_FIXED = frozenset(LEAKS)


def build_protocol(book: Book, fixed: frozenset, seed: int):
    """Turn a set of fixed leaks into (book, folds, pipeline, used_columns). Given."""
    if "duplicate" in fixed:
        book = dedupe(book)
    used = tuple(n for n in NAMES if not ("target" in fixed and n == LATE_FEATURE))
    in_fold = "preprocessing" in fixed
    pipeline = make_pipeline(used, scaler_in_fold=in_fold, encoder_in_fold=in_fold)
    folds = protocol_folds(book, "group" in fixed, "temporal" in fixed, seed)
    return book, folds, pipeline, used


def cv_auc(book: Book, folds: list, pipeline) -> float:
    """Mean over folds of the AUC on each fold's test rows. Given."""
    return float(np.mean([auc_score(book.y[te], pipeline(book.X, book.y, tr, te))
                          for tr, te in folds]))


_AUC_CACHE: dict = {}


def evaluate(book_index: int, fixed: frozenset) -> float:
    """Cross-validated AUC of one book under one protocol, remembered. Given."""
    key = (book_index, frozenset(fixed))
    if key not in _AUC_CACHE:
        book, folds, pipeline, _ = build_protocol(BOOKS[book_index], frozenset(fixed), book_index)
        _AUC_CACHE[key] = cv_auc(book, folds, pipeline)
    return _AUC_CACHE[key]


def true_score(X: np.ndarray) -> np.ndarray:
    """The planted truth as a score: the usage signals with their true weights. Given. On average a
    ceiling for any model that uses only the usage signals, because the customer and month effects
    it leaves out cannot be known for a new customer."""
    return X[:, 0] * COEF[0] + X[:, 1] * COEF[1] + X[:, 2] * COEF[2]


def _signed(x: float) -> str:
    """+0.012 or -0.012, and +0.000 rather than -0.000 for a value that rounds to zero."""
    v = round(float(x), 3)
    return f"{(v if v != 0 else 0.0):+.3f}"


def _se(values) -> float:
    v = np.asarray(values, dtype=float)
    return float(v.std(ddof=1) / math.sqrt(len(v))) if len(v) > 1 else 0.0

In [ ]:
HEADLINE: dict[str, list[float]] = {}


def _headline() -> None:
    HEADLINE.update(naive=[], honest=[], naive_ceiling=[], honest_ceiling=[])
    for i in range(N_BOOKS):
        HEADLINE["naive"].append(evaluate(i, frozenset()))
        HEADLINE["honest"].append(evaluate(i, ALL_FIXED))
        for key, fixed in (("naive_ceiling", frozenset()), ("honest_ceiling", ALL_FIXED)):
            book, folds, _, _ = build_protocol(BOOKS[i], fixed, i)      # the ceiling on THIS protocol's test rows
            HEADLINE[key].append(float(np.mean([auc_score(book.y[te], true_score(book.X[te]))
                                                for _, te in folds])))
    print(f"cross-validated AUC, mean over {N_BOOKS} books (standard error in brackets)")
    for label, key in (("naive protocol (nothing fixed)", "naive"),
                       ("  oracle ceiling, same rows (usage only)", "naive_ceiling"),
                       ("honest protocol (all five fixed)", "honest"),
                       ("  oracle ceiling, same rows (usage only)", "honest_ceiling")):
        v = HEADLINE[key]
        print(f"  {label:<44} {np.mean(v):.3f}  ({_se(v):.3f})")
    over = np.array(HEADLINE["naive"]) - np.array(HEADLINE["naive_ceiling"])
    gap = np.array(HEADLINE["honest"]) - np.array(HEADLINE["honest_ceiling"])
    print(f"naive minus its ceiling: {over.mean():+.3f} (standard error {_se(over):.3f}); "
          f"honest minus its ceiling: {gap.mean():+.3f} (standard error {_se(gap):.3f})")
    if over.mean() > 2 * _se(over):
        print("The naive score is above what a model using only the usage signals can earn, on average, "
              "on new customers here: the surplus did not come from the usage signals, so, for a model "
              "that will score new customers, it came from somewhere the protocol should not have let it look.")
    else:
        print("The naive score is not clearly above the ceiling on these books.")
    print("The two protocols differ only in the five fixes in the table above; the rest of this lesson "
          "finds what each one is worth.")


if _DEMOS:
    _try("naive versus honest", _headline)

## 3. Five leaks, five tells

A leak is information from the test rows (or from outside the prediction moment) reaching the
model. Each of the five leaks leaves a different **tell** that a check can look for *without
measuring a cross-validated score*: two of the checks read only the data, and the third only re-runs
the pipeline on altered inputs. That is why a clinic is cheaper than a post-mortem:

| leak | what to compare | the check you will write |
|---|---|---|
| duplicate | the rows' keys, training against test | `split_leaks`, count of repeated keys |
| group | the customers, training against test | `split_leaks`, count of shared groups |
| temporal | the prediction dates, training against test | `split_leaks`, count of rows too early |
| target | when each value was written, against when the row was scored | `feature_audit` |
| preprocessing | what the scores do when test labels or other test rows change | `fold_isolation` |

Two things to hold on to. A check reports whether a leak is **possible**, not whether it cost
you anything: a split can break a rule and still lose nothing on the data in hand, and the
measurements in section 8 are what tell you how much a real leak is worth. And fixing one leak
does not fix the others, because one memorised column (`postcode` here) can leak through the
rows, the customers and the preprocessing at once.

**Standard tooling** (`claims.yaml`, scikit-learn 1.9.1 documentation as read on 14 October 2026).
`GroupKFold` keeps each group wholly on one side of a split and `TimeSeriesSplit` trains on
ever-longer stretches of the past; the guide's advice for preprocessing is never to include test
data when fitting. This lesson writes its own versions so that it needs only numpy.

**In the wild** (sources and access dates in `claims.yaml`, as of 15 October 2026). A survey of
machine-learning-based science (Kapoor and Narayanan, *Patterns*, 2023) found leakage errors in 17
fields, collectively affecting 294 papers; the authors' running list, last updated in May 2024,
counts 648 papers in 30 fields, so treat any such count as a snapshot. The survey's taxonomy
includes pre-processing on the training and test sets together, duplicates, features that are not
legitimately available, temporal leakage, and non-independence between train and test samples
(leakage unless the claim is about a distribution with the same dependence structure). On the CIFAR
image benchmarks, 3.3% and 10% of the test images of CIFAR-10 and CIFAR-100 had duplicates or
near-duplicates in the training set; this lesson plants only exact copies.

In [ ]:
if _DEMOS:
    b = BOOKS[0]
    ids, per_customer = np.unique(b.customer, return_counts=True)
    j = NAMES.index(LATE_FEATURE)
    lag = (b.written_at[:, j] - b.predicted_at)[b.written_at[:, j] > b.predicted_at]
    print("the raw material each check will look at, on book 0 (none of this is a check yet)")
    print(f"  duplicate:     {len(b.keys) - len(set(b.keys))} rows have a key that an earlier row already has")
    print(f"  group:         {int((per_customer > 1).sum())} of {len(ids)} customer ids have more than one row")
    print(f"  temporal:      prediction days run from {int(b.predicted_at.min())} to "
          f"{int(b.predicted_at.max())}; the label looks {HORIZON_DAYS} days ahead")
    print(f"  target:        {LATE_FEATURE} was written {int(lag.min())} to {int(lag.max())} days after the "
          f"prediction day, in {len(lag)} rows")
    print("  preprocessing: nothing to read off the book; it is a property of the pipeline's code")

## 4. Exercise 1 — `split_leaks()`: what crosses the boundary of one fold?

Three of the five leaks are properties of a *split*, so they can be audited from the fold's row
numbers alone, before any model is fitted. For one fold (a list of training rows and a list of
test rows) count:

* **group**: how many distinct groups (customers) have rows on both sides;
* **time**: how many test rows are **not at least `gap` after the latest training row**, that
  is, test rows with `times[row] < latest training time + gap`. The gap is there because the
  label of a training row at day `t` is only known at `t + HORIZON_DAYS`;
* **duplicate**: how many test rows have a **key equal to the key of some training row**. A row
  that repeats another test row, or a training row, only on its own side, leaks nothing across.
  A key is any hashable Python value (a string, bytes or a tuple, say), compared as a value.

These are **counts**, not yes or no, because "3 customers straddle the boundary" and "3,000 do"
call for different reactions. Two details to get right: a group is counted once however many
rows it has, and a test row exactly `gap` after the latest training row is fine (the rule is a
strict "earlier than"). With no training rows there is nothing to leak from: all three counts
are 0.

<details><summary>💡 Hint 1 — what to think about</summary>

Three independent questions, three different things to compare. For the first, what is the
right unit to count, rows or groups? For the second, which single training row decides whether
a test row is too early? For the third, what lookup makes "does this key occur on the training
side?" fast, and what happens to a key that occurs only among the test rows?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Groups: collect the group ids of the training rows and of the test rows, and count the ids that
appear in both collections. Time: find the latest training time, add the gap, and count the test
rows whose time is below that. Duplicate: put the training rows' keys in a set and count the test
rows whose key is in it. Return the three counts under their three names.
</details>

In [ ]:
def split_leaks(train_idx, test_idx, groups, times, keys, gap):
    """Count what crosses from the training rows into the test rows of one fold.

    `train_idx` and `test_idx` are row numbers (lists or numpy arrays); `groups[i]`, `times[i]` and
    `keys[i]` describe row i, and a key is any hashable value.

    Example:
        >>> split_leaks([0, 1], [2, 3], groups=[7, 7, 8, 9], times=[0, 5, 6, 9],
        ...             keys=["a", "b", "c", "a"], gap=3)
        {'group': 0, 'time': 1, 'duplicate': 1}
        (no customer is on both sides; the test row at day 6 is before 5 + 3, the one at day 9
        is not; row 3 repeats row 0's key)

    Returns:
        {"group": int, "time": int, "duplicate": int}, the three counts defined above.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_split_leaks() -> None:
    got = split_leaks([0, 1], [2, 3], [7, 7, 8, 9], [0, 5, 6, 9], ["a", "b", "c", "a"], 3)
    assert got == {"group": 0, "time": 1, "duplicate": 1}, (
        f"got {got}, expected {{'group': 0, 'time': 1, 'duplicate': 1}}: the worked example in the "
        "docstring has no shared customer, one test row before day 5 + 3 and one repeated key"
    )
    got = split_leaks([0, 1, 2], [3, 4, 5], [5, 5, 6, 5, 5, 7], [0, 1, 2, 3, 4, 5], list(range(6)), 0)
    assert got["group"] == 1, (
        f"got {got['group']} shared groups, expected 1: customer 5 has two rows in the test fold "
        "and two in the training fold but is still ONE customer on both sides; count groups, not rows"
    )
    on_edge = split_leaks([0], [1, 2], [0, 1, 2], [10, 12, 11], [0, 1, 2], 2)
    assert on_edge["time"] == 1, (
        f"got {on_edge['time']} rows too early, expected 1: with the latest training row at day 10 "
        "and a gap of 2, day 11 is too early and day 12 is fine; the rule is a strict 'earlier than'"
    )
    twins_in_test = split_leaks([0], [1, 2], [0, 1, 2], [0, 40, 50], ["x", "y", "y"], 5)
    assert twins_in_test["duplicate"] == 0, (
        "two test rows that share a key with each other but with no training row leak nothing: "
        "only a key that also occurs among the training rows counts"
    )
    empty = split_leaks([], [0, 1], [3, 4], [1, 2], ["a", "b"], 7)
    assert empty == {"group": 0, "time": 0, "duplicate": 0}, (
        f"got {empty}: with no training rows there is nothing to leak from, so every count is 0"
    )
    rows = np.array([0, 1, 2, 3])
    as_arrays = split_leaks(rows[:2], rows[2:], np.array([7, 7, 8, 9]), np.array([0, 5, 6, 9]),
                            ["a", "b", "c", "a"], 3)
    assert as_arrays == {"group": 0, "time": 1, "duplicate": 1}, (
        f"got {as_arrays} when the row numbers, groups and times are numpy arrays: every demo below "
        "passes arrays, so a test such as `if not train_idx` (which fails on an array) must not decide anything"
    )
    none_yet = split_leaks(rows[:0], rows[2:], np.array([7, 7, 8, 9]), np.array([0, 5, 6, 9]),
                           ["a", "b", "c", "a"], 3)
    assert none_yet == {"group": 0, "time": 0, "duplicate": 0}, (
        f"got {none_yet} for an empty numpy array of training rows: every count is 0"
    )
    print("exercise 1 looks right")


_try("exercise 1", _check_split_leaks)

In [ ]:
def _split_audit() -> None:
    families = (
        ("random rows", frozenset()),
        ("random rows, repeats dropped first", frozenset({"duplicate"})),
        ("whole customers (group k-fold)", frozenset({"group"})),
        ("forward in time, by row", frozenset({"temporal"})),
        ("forward in time, by customer cohort", frozenset({"group", "temporal"})),
        ("repeats dropped, customers, forward", frozenset({"duplicate", "group", "temporal"})),
    )
    print(f"book 0, counts summed over each family's folds (gap = HORIZON_DAYS = {HORIZON_DAYS})")
    print(f"  {'folds':<40}{'group':>7}{'time':>7}{'duplicate':>11}   test rows")
    totals = {}
    for label, fixed in families:
        book, folds, _, _ = build_protocol(BOOKS[0], fixed, 0)
        total = {"group": 0, "time": 0, "duplicate": 0}
        for train, test in folds:
            got = _call("split_leaks", split_leaks, train, test, book.customer, book.predicted_at,
                        book.keys, HORIZON_DAYS)
            for k in total:
                total[k] += got[k]
        totals[label] = total
        n_test = sum(len(te) for _, te in folds)
        print(f"  {label:<40}{total['group']:>7}{total['time']:>7}{total['duplicate']:>11}   {n_test}")
    silent = [label for label, total in totals.items() if not any(total.values())]
    print("silent on all three counts: " + (", ".join(silent) if silent else "none"))
    cohort = totals["forward in time, by customer cohort"]
    repeats = len(BOOKS[0].keys) - len(set(BOOKS[0].keys))
    if not any(cohort.values()) and repeats:
        print(f"The cohort-forward folds are silent although book 0 still holds {repeats} repeated rows: "
              "a copy has the same prediction day as its original, so both fall inside the same window "
              "and stay on the same side. The audit reports what the FOLDS expose, which can be less "
              "than the mistakes left in the recipe.")
    rand, whole = totals["random rows"], totals["whole customers (group k-fold)"]
    print(f"Read across the rows: dropping repeats first takes the duplicate count from {rand['duplicate']} to "
          f"{totals['random rows, repeats dropped first']['duplicate']}, and keeping customers whole takes the "
          f"group count from {rand['group']} to {whole['group']}; each count answers one question about the folds.")
    if whole["duplicate"]:
        print("Keeping customers whole leaves the duplicate count above zero: a copy is registered under a "
              "new customer id, so a split by customer can still put a copy and its original on different sides.")


if _DEMOS:
    _try("split audit", _split_audit, needs=("exercise 1",))

## 5. Exercise 2 — `feature_audit()`: was this column known when the row was scored?

The target leak is not a property of the split but of a **column**, and the split audit cannot
see it. The strongest check is a **point-in-time** one: every value in the record carries the
day it was written, every row carries the day it was scored, and a column that was written
*after* the day its row was scored could not have been used by the deployed model. This is the
definition of leakage in scikit-learn's guide (information that would not be available at
prediction time) and what a feature store's "point-in-time correct join" enforces (`claims.yaml`).

A second, weaker check needs no timestamps: the **too-good-to-be-true scan**. A single column
whose AUC on its own is close to 1 (or close to 0, which is the same signal upside down) is
suspicious. It finds loud leaks and misses quiet ones, and the demo shows both.

Write the audit of both, on the columns of `X`:

* **late**: a column is late when MORE THAN `max_late` of its rows have
  `written_at[row, col] > predicted_at[row]`. Equal is not late: a value stamped the same day as
  the prediction was available that day. Compare with each row's own prediction day.
* **too_good**: with `a` the column's AUC on its own (use `auc_score(y, X[:, col])`), a column
  is flagged when `max(a, 1 - a)`, **rounded to 6 decimals**, is at least `auc_bound`. Rounding
  first keeps a column at exactly the bound from being missed by a float edge. If `y` holds a
  single class there is no AUC and nothing is flagged.

Both lists keep the order of `names`.

<details><summary>💡 Hint 1 — what to think about</summary>

Two loops over the columns, or one loop doing two jobs. For lateness, which array do you compare
a column of `written_at` against, and is it one number or one per row? For the scan, what is a
column's AUC when it is exactly uninformative, and why does the upside-down case need its own
handling?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

For each column, count the rows whose written day is strictly after their own prediction day and
flag the column when that count is above `max_late`. Separately, compute the column's AUC, take the
larger of it and one minus it, round to six places, and flag the column when that reaches the
bound; skip the scan when `y` has only one class.
</details>

In [ ]:
def feature_audit(names, X, y, written_at, predicted_at, max_late=0, auc_bound=AUC_BOUND):
    """Audit the columns of X for lateness and for being too good to be true.

    `names[j]` is column j; `X` is (rows, columns); `written_at` is (rows, columns) days;
    `predicted_at` is (rows,) days; `y` is 0/1. Rules are in the text above.

    Example:
        >>> X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
        >>> y = np.array([0, 0, 1, 1])
        >>> written = np.array([[1, 1], [1, 4], [1, 1], [1, 1]])
        >>> feature_audit(["a", "b"], X, y, written, np.array([1, 1, 1, 1]))
        {'late': ['b'], 'too_good': ['a']}
        (column b was written at day 4 for a row scored at day 1; column a separates y perfectly)

    Returns:
        {"late": [names], "too_good": [names]}, each in the order of `names`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_feature_audit() -> None:
    X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
    y = np.array([0, 0, 1, 1])
    written = np.array([[1, 1], [1, 4], [1, 1], [1, 1]])
    got = feature_audit(["a", "b"], X, y, written, np.array([1, 1, 1, 1]))
    assert got == {"late": ["b"], "too_good": ["a"]}, (
        f"got {got}, expected {{'late': ['b'], 'too_good': ['a']}}: the docstring example has one "
        "column written after its row was scored and one that separates y perfectly"
    )
    same_day = feature_audit(["a"], X[:, :1], y, np.array([[5], [5], [5], [5]]), np.array([5, 5, 5, 5]))
    assert same_day["late"] == [], (
        "a value written on the very day the row was scored was available that day: the lateness "
        "test is a strict 'after', not 'on or after'"
    )
    per_row = feature_audit(["a"], X[:, :1], y, np.array([[10], [10], [3], [3]]), np.array([20, 20, 2, 2]))
    assert per_row["late"] == ["a"], (
        "compare each row's written day with that row's OWN prediction day: rows 2 and 3 were "
        "written at day 3 for rows scored at day 2"
    )
    allowed = feature_audit(["a"], X[:, :1], y, np.array([[9], [1], [1], [1]]), np.array([5, 5, 5, 5]), max_late=1)
    assert allowed["late"] == [], "one late row is allowed when max_late is 1: flag only MORE THAN max_late rows"
    flipped = feature_audit(["a"], 1.0 - X[:, :1], y, np.array([[1]] * 4), np.array([1, 1, 1, 1]))
    assert flipped["too_good"] == ["a"], (
        "a column with AUC near 0 separates the classes just as well upside down: flag max(a, 1 - a)"
    )
    print("exercise 2 looks right")


_try("exercise 2", _check_feature_audit)

In [ ]:
def _feature_demo() -> None:
    book = BOOKS[0]
    got = _call("feature_audit", feature_audit, NAMES, book.X, book.y, book.written_at, book.predicted_at)
    print(f"late columns:     {got['late']}")
    print(f"too-good columns: {got['too_good']}  (bound {AUC_BOUND})")
    j = NAMES.index(LATE_FEATURE)
    alone = auc_score(book.y, book.X[:, j])
    same_day = int((book.written_at[:, NAMES.index('month')] == book.predicted_at).sum())
    print(f"{LATE_FEATURE} on its own has AUC {alone:.3f}; month is stamped the same day as its row "
          f"in {same_day} of {len(book.y)} rows and is "
          + ("not flagged late." if "month" not in got["late"] else "flagged late: check the strict '>'."))
    if LATE_FEATURE in got["late"] and LATE_FEATURE not in got["too_good"]:
        print(f"The timestamps caught {LATE_FEATURE}; the scan did not, because its AUC on its own, "
              f"{alone:.3f}, is below the bound. A partial leak looks like a merely good feature.")
    loud = np.column_stack([book.X, book.y.astype(float)])
    loud_written = np.column_stack([book.written_at, book.predicted_at + HORIZON_DAYS])
    got2 = _call("feature_audit", feature_audit, NAMES + ("cancelled_flag",), loud, book.y,
                 loud_written, book.predicted_at)
    print(f"adding a column that IS the label, written {HORIZON_DAYS} days after scoring: "
          f"late {got2['late']}, too good {got2['too_good']}")
    if "cancelled_flag" in got2["late"] and "cancelled_flag" in got2["too_good"]:
        print("The added label column trips both audits.")
    if LATE_FEATURE in got2["late"] and LATE_FEATURE not in got2["too_good"]:
        print(f"{LATE_FEATURE} trips only the audit that knows when each value was written, which is "
              "why a record of when every field was written is worth keeping.")


if _DEMOS:
    _try("feature audit", _feature_demo, needs=("exercise 2",))

## 6. Exercise 3 — `fold_isolation()`: does the pipeline peek?

The preprocessing leak is a property of the **pipeline**: some step was fitted using rows or
labels it should not have seen. You cannot see that by reading the split, and a leak like it can
hide anywhere in a long pipeline. But it has a tell you can test without reading a line of the
pipeline: **a prediction for a test row may depend on the training fold and on that row, and
on nothing else.** So change something it should not depend on, and see whether the scores move.

A pipeline is given as `fit_predict(X, y, train_idx, test_idx)`, which fits on the training rows
and returns one score per test row. Run it three times and compare, calling a violation any score
that moves by MORE than `ISOLATION_TOL` (an absolute difference, whatever the size of the scores;
a move of exactly `ISOLATION_TOL` is not a violation):

* **test_labels**: flip every test label (`y` becomes `1 - y` at the test rows), run again, and
  compare ALL the scores. A pipeline that has not read the test labels cannot notice.
* **other_test_rows**: let `h = (len(test) + 1) // 2`. Keep the first `h` test rows; replace each
  remaining test row's features with the features of a training row, by exactly this rule: the
  k-th replaced row (k counts from 0, in the order of `test`) gets the features of
  `train[k % len(train)]`. Run again, and compare only the scores of the first `h` rows. A
  pipeline that reads other test rows (to standardise, to rank, to encode) moves them. With fewer
  than two test rows, or no training rows, this second test cannot be run and reports False; a
  fold with no test rows reports False twice.

<details><summary>💡 Hint 1 — what to think about</summary>

Never modify the arrays you were given: copy before you change anything. For the second test,
which scores do you compare, and why would comparing every score give a false alarm on a perfectly
honest pipeline? What is the baseline each of the two runs is compared with?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Get the baseline scores once. For labels, copy `y`, flip the labels at the test rows, run the
pipeline again with the flipped copy and compare the whole score vector with the baseline. For
the other test rows, copy `X`, overwrite the rows after the first half with training-row features
(the rule above says which training row each one takes, and wraps round when the training fold is
shorter than the number of rows to replace), run again, and compare only the first half of the
scores. Report each comparison as a bool.
</details>

In [ ]:
def fold_isolation(fit_predict, X, y, train_idx, test_idx):
    """Test whether a pipeline's scores on the test fold depend on what they should not.

    `fit_predict(X, y, train_idx, test_idx)` returns one score per test row. Rules are above.

    Example:
        >>> X = np.arange(12.0).reshape(6, 2)
        >>> y = np.array([0, 1, 0, 1, 1, 0])
        >>> honest = lambda X, y, tr, te: X[te, 0] * 2 + y[tr].mean()
        >>> peeks = lambda X, y, tr, te: X[te, 0] * 2 + y[te].mean()
        >>> fold_isolation(honest, X, y, [0, 1, 2], [3, 4, 5])
        {'test_labels': False, 'other_test_rows': False}
        >>> fold_isolation(peeks, X, y, [0, 1, 2], [3, 4, 5])
        {'test_labels': True, 'other_test_rows': False}

    Returns:
        {"test_labels": bool, "other_test_rows": bool}
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_fold_isolation() -> None:
    X = np.arange(12.0).reshape(6, 2)
    y = np.array([0, 1, 0, 1, 1, 0])
    honest = lambda X, y, tr, te: X[te, 0] * 2 + y[tr].mean()
    peeks = lambda X, y, tr, te: X[te, 0] * 2 + y[te].mean()
    scaled = lambda X, y, tr, te: X[te, 0] - X[:, 0].mean()
    got = fold_isolation(honest, X, y, [0, 1, 2], [3, 4, 5])
    assert got == {"test_labels": False, "other_test_rows": False}, (
        f"got {got} for a pipeline that only reads the training labels and each test row's own "
        "features: it must report False twice. Compare only the scores of the rows you did NOT "
        "change in the second test, or an honest pipeline is accused because its changed rows moved"
    )
    got = fold_isolation(peeks, X, y, [0, 1, 2], [3, 4, 5])
    assert got["test_labels"] is True, (
        "this pipeline's scores depend on the mean of the TEST labels: flipping them must move the "
        "scores, so test_labels must be True"
    )
    as_arrays = fold_isolation(honest, X, y, np.array([0, 1, 2]), np.array([3, 4, 5]))
    assert as_arrays == {"test_labels": False, "other_test_rows": False}, (
        f"got {as_arrays} when the row numbers are numpy arrays: every demo below passes arrays, so "
        "build the replacement rows with array indexing, not list arithmetic"
    )
    got = fold_isolation(scaled, X, y, [0, 1, 2], [3, 4, 5])
    assert got == {"test_labels": False, "other_test_rows": True}, (
        f"got {got}: this pipeline centres by the mean of ALL rows, test rows included, so changing "
        "other test rows must move the first rows' scores (other_test_rows True) and the labels "
        "play no part (test_labels False)"
    )
    X_before, y_before = X.copy(), y.copy()
    fold_isolation(peeks, X, y, [0, 1, 2], [3, 4, 5])
    assert (X == X_before).all() and (y == y_before).all(), (
        "fold_isolation changed the arrays it was given: work on copies"
    )
    X8, runs = np.arange(16.0).reshape(8, 2), []

    def recorder(X, y, tr, te):
        runs.append(X.copy())
        return X[te, 0] * 1.0

    fold_isolation(recorder, X8, np.zeros(8, dtype=int), [0, 1], [2, 3, 4, 5, 6, 7])
    assert len(runs) == 3, f"the pipeline is run three times (baseline, flipped labels, replaced rows); got {len(runs)}"
    assert (runs[2][:5] == X8[:5]).all() and (runs[2][5:] == X8[[0, 1, 0]]).all(), (
        f"with 6 test rows the last 3 are replaced; they now hold {runs[2][5:].tolist()}, but the k-th "
        "replaced row (k from 0) must take the features of train[k % len(train)], here training rows 0, 1, 0: "
        "a training fold shorter than the number of replaced rows is reused from its start"
    )
    print("exercise 3 looks right")


_try("exercise 3", _check_fold_isolation)

In [ ]:
def _isolation_demo() -> None:
    used = tuple(n for n in NAMES if n != LATE_FEATURE)
    variants = (
        ("honest pipeline", dict(scaler_in_fold=True, encoder_in_fold=True)),
        ("scaler fitted on train + test rows", dict(scaler_in_fold=False, encoder_in_fold=True)),
        ("encoders fitted on every row's label", dict(scaler_in_fold=True, encoder_in_fold=False)),
    )
    results: dict[str, dict] = {}
    for label, options in variants:
        pipeline = make_pipeline(used, **options)
        flagged = {"test_labels": 0, "other_test_rows": 0}
        n_folds, aucs = 0, []
        for i in range(N_BOOKS):
            book, folds, _, _ = build_protocol(BOOKS[i], ALL_FIXED, i)
            for train, test in folds:
                got = _call("fold_isolation", fold_isolation, pipeline, book.X, book.y, train, test)
                for k in flagged:
                    flagged[k] += int(got[k])
                n_folds += 1
            aucs.append(cv_auc(book, folds, pipeline))
        results[label] = {"flagged": flagged, "n": n_folds, "auc": np.array(aucs)}
    honest = results["honest pipeline"]["auc"]
    print("honest forward folds, deduplicated, retention_offer left out; every book and fold")
    print(f"  {'pipeline':<40}{'labels moved it':>17}{'other rows moved it':>21}{'AUC':>8}"
          f"{'gap to honest':>16}")
    for label, _ in variants:
        r = results[label]
        gap = r["auc"] - honest
        print(f"  {label:<40}{r['flagged']['test_labels']:>10} of {r['n']:<3}"
              f"{r['flagged']['other_test_rows']:>13} of {r['n']:<3}{r['auc'].mean():>8.3f}"
              f"{_signed(gap.mean()):>10} ({_se(gap):.3f})")
    for label, _ in variants[1:]:
        r = results[label]
        gap = r["auc"] - honest
        caught = any(r["flagged"].values())
        costly = gap.mean() > 2 * _se(gap)
        print(f"{label}: " + ("caught by the check" if caught else "NOT caught by the check")
              + ", and " + ("it inflates AUC beyond two standard errors." if costly else
                             "its AUC gap is within two standard errors of zero."))
    print("The check says a leak is possible; only the AUC gap says what it costs on these books.")


if _DEMOS:
    _try("fold isolation", _isolation_demo, needs=("exercise 3",))

## 7. Exercise 4 — `diagnose()`: one report for a whole protocol

You now have three checks. A **diagnosis** puts them together: given a book, the folds of a
protocol and its pipeline, name the leaks the protocol exposes. Mapping from check to leak:

| finding | leak named |
|---|---|
| `feature_audit` finds a late or too-good column, among the columns in `used` | `"target"` |
| any fold has `split_leaks(...)["time"] > 0` | `"temporal"` |
| any fold has `split_leaks(...)["group"] > 0` | `"group"` |
| any fold has `split_leaks(...)["duplicate"] > 0` | `"duplicate"` |
| any fold has `fold_isolation(...)` True on either key | `"preprocessing"` |

Three rules the clinic needs. **Every fold**, not just the first: a leak in the last fold is a
leak. **Only the columns the pipeline reads** (`used`): a late column nobody uses is not a leak.
**No false alarms**: an honest protocol returns an empty list. Return the leak names in the order
of `LEAKS` (target, preprocessing, duplicate, group, temporal), each at most once.
`split_leaks` takes the customer ids as groups, `predicted_at` as times and `book.keys` as keys.

`used` narrows the **feature audit** and nothing else. The pipeline picks its own columns by
position in the book's `X`, so `fold_isolation` is always given the whole `book.X`, never a copy
cut down to the used columns.

<details><summary>💡 Hint 1 — what to think about</summary>

Which of the five findings is about the whole book, and which are about one fold at a time? A
leak can appear in only one of several folds; how will you make sure it is not lost? How do you
restrict a column audit to the columns the pipeline reads, given only their names?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Find the positions of the used columns in `book.names`, cut `X` and `written_at` to them for the
feature audit only, and run that audit once. Then loop over the folds: run the split audit on
each, and run the isolation test on each with the whole `book.X` (the pipeline picks its own
columns by position, so it must not be handed the cut-down copy). Record each leak in a set when
any count is positive or any flag is True. Finish by listing the leaks in `LEAKS` order that are
in the set.
</details>

In [ ]:
def diagnose(book, folds, fit_predict, used, gap=HORIZON_DAYS, max_late=0, auc_bound=AUC_BOUND):
    """Name the leaks that a protocol (a book, its folds and its pipeline) exposes.

    `book` is a Book, `folds` a list of (train_idx, test_idx), `fit_predict` the pipeline as in
    exercise 3, and `used` the names of the columns the pipeline reads. Uses your split_leaks,
    feature_audit and fold_isolation. `used` narrows the feature audit only: fold_isolation is
    given the whole `book.X`.

    Example:
        >>> book, folds, pipeline, used = build_protocol(BOOKS[0], frozenset(), 0)
        >>> diagnose(book, folds, pipeline, used)
        ['target', 'preprocessing', 'duplicate', 'group', 'temporal']

    Returns:
        A list of leak names, in the order of LEAKS, each at most once; [] for a clean protocol.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _toy_book(late_in_f2: bool = False) -> Book:
    """Eight rows, two columns: f1 is honest; f2 is written late when asked. Used by the checks."""
    n = 8
    day = np.arange(n) * 100
    written = np.column_stack([day - 1, day + (4 if late_in_f2 else -1)])
    return Book(("f1", "f2"), np.column_stack([np.arange(n) % 3, np.arange(n) % 2]).astype(float),
                np.array([0, 1, 0, 1, 1, 0, 1, 0]), np.array([0, 0, 1, 1, 2, 2, 3, 3]),
                day, written, [f"row{i}" for i in range(n)])


# Public checks — run these as often as you like.
def _check_diagnose() -> None:
    clean_folds = [(np.array([0, 1, 2, 3]), np.array([5, 6])), (np.array([0, 1, 2, 3, 4]), np.array([7]))]
    honest = lambda X, y, tr, te: X[te, 0] * 1.0 + y[tr].mean()
    book = _toy_book()
    hint = ("run the three public checks above first; diagnose is only as good as the checks it "
            "calls, and an honest protocol must give []")
    got = diagnose(book, clean_folds, honest, ("f1", "f2"))
    assert got == [], f"got {got} for a clean toy protocol (forward folds, own customers, own keys, honest pipeline): {hint}"
    got = diagnose(_toy_book(late_in_f2=True), clean_folds, honest, ("f1", "f2"))
    assert got == ["target"], f"got {got}: column f2 is written after its row was scored, so the answer is ['target']"
    got = diagnose(_toy_book(late_in_f2=True), clean_folds, honest, ("f1",))
    assert got == [], f"got {got}: the late column f2 is not among the columns this pipeline uses, so it is not a leak"
    straddling = [(np.array([0, 1, 2, 3]), np.array([5, 6])), (np.array([0, 1, 2]), np.array([3, 7]))]
    got = diagnose(book, straddling, honest, ("f1", "f2"))
    assert "group" in got, (
        f"got {got}: the SECOND fold has customer 1 on both sides (rows 2 and 3); every fold must "
        "be audited, not just the first"
    )
    wide = Book(("f1", "f2", "f3"), np.column_stack([book.X, np.arange(8) % 4]).astype(float), book.y,
                book.customer, book.predicted_at, np.column_stack([book.written_at, book.predicted_at - 1]),
                book.keys)
    third = lambda X, y, tr, te: X[te, 2] + y[tr].mean()          # reads column 2 of the book's X by position
    try:
        got = diagnose(wide, clean_folds, third, ("f1", "f3"))
    except IndexError as exc:
        raise AssertionError(
            f"diagnose raised IndexError ({exc}) for a pipeline that reads column 2 of the book's X: "
            "`used` narrows the feature audit only, so give fold_isolation the WHOLE book.X, not a copy cut down to the used columns"
        ) from None
    assert got == [], (
        f"got {got} for an honest pipeline that reads column 2 of the book's X while `used` is ('f1', 'f3'): "
        "give fold_isolation the WHOLE book.X; only the feature audit is cut down to the used columns"
    )
    print("exercise 4 looks right")


_try("exercise 4", _check_diagnose, needs=("exercise 1", "exercise 2", "exercise 3"))

In [ ]:
def _ladder() -> None:
    print("Fixing the leaks one at a time, in this order: " + ", ".join(LEAKS))
    print(f"  {'fixed so far':<14}{'the audit finds':<64}{'AUC':>6}{'removed by the last fix':>28}")
    previous, matched, pairs = None, 0, 0
    for r in range(len(LEAKS) + 1):
        fixed = frozenset(LEAKS[:r])
        shown = None
        for i in range(N_BOOKS):
            book, folds, pipeline, used = build_protocol(BOOKS[i], fixed, i)
            found = _call("diagnose", diagnose, book, folds, pipeline, used)
            matched += int(found == [leak for leak in LEAKS if leak not in fixed])
            pairs += 1
            shown = found if shown is None else shown
        aucs = np.array([evaluate(i, fixed) for i in range(N_BOOKS)])
        step = "" if previous is None else f"{_signed((previous - aucs).mean())} ({_se(previous - aucs):.3f})"
        label = "none" if r == 0 else LEAKS[r - 1]
        print(f"  {label:<14}{str(shown):<64}{aucs.mean():>6.3f}{step:>28}")
        previous = aucs
    print(f"At every rung the audit named exactly the leaks not yet fixed in {matched} of {pairs} "
          "(rung, book) pairs.")
    if matched == pairs:
        print("So each check went quiet when, and only when, its own leak was fixed, in this order.")
    else:
        print("In some rung the audit and the list of unfixed leaks differ: the audit reports what the "
              "FOLDS expose, which can differ from the mistakes left in the recipe.")


if _DEMOS:
    _try("ladder", _ladder, needs=("exercise 1", "exercise 2", "exercise 3", "exercise 4"))

## 8. Exercise 5 — `average_steps()`: what is each leak worth?

The ladder above attributes to each fix the AUC it removed **in that order**. That number is not
a property of the leak: remove the same leak later, after other leaks are gone, and it can remove
something different, because leaks overlap (one memorised column leaks through several routes).

A fairer number averages a leak's contribution over **every order** in which the leaks could be
added to an honest protocol. Give it `score`, a dictionary from each frozenset of leaks that are
**present** (the other leaks fixed) to the cross-validated AUC of that protocol. Then the step of a
leak in one order is `score[present | {leak}] - score[present]`, where `present` is the set of
leaks that came before it in that order, and its **average step** is the mean of those steps over
all `len(names)!` orders. (This is the Shapley value of the game "leaks inflate the score".) The
steps of one order add up to `score[all] - score[none]`, and so do the averages.

<details><summary>💡 Hint 1 — what to think about</summary>

An order is a permutation of the names. Walking along one permutation, what is the set of leaks
present just before each name is reached, and just after? How many permutations are there for five
names, and is it small enough to enumerate?
</details>
<details><summary>💡 Hint 2 — the approach, in words</summary>

Keep a running total per name. For each permutation, start with an empty set of present leaks, and
for each name in turn add it to the set, record the change in `score` between the old and the new
set, and add that change to the name's total. At the end divide every total by the number of
permutations.
</details>

In [ ]:
def average_steps(score, names):
    """The average, over every order of `names`, of each name's step in `score`.

    `score[frozenset_of_present_names]` is the AUC with exactly those leaks present. Every subset of
    `names` is a key.

    Example:
        >>> both = {frozenset(): 0.0, frozenset({"a"}): 0.0, frozenset({"b"}): 0.0,
        ...         frozenset({"a", "b"}): 1.0}
        >>> average_steps(both, ["a", "b"])
        {'a': 0.5, 'b': 0.5}
        (neither leak does anything alone; together they add 1.0, shared equally over the two orders)

    Returns:
        {name: average step}, in the order of `names`.
    """
    # YOUR CODE HERE
    raise NotImplementedError


# Public checks — run these as often as you like.
def _check_average_steps() -> None:
    both = {frozenset(): 0.0, frozenset({"a"}): 0.0, frozenset({"b"}): 0.0, frozenset({"a", "b"}): 1.0}
    got = average_steps(both, ["a", "b"])
    assert got == {"a": 0.5, "b": 0.5}, (
        f"got {got}, expected {{'a': 0.5, 'b': 0.5}}: each name comes first in half of the orders "
        "(step 0) and second in the other half (step 1)"
    )
    additive = {frozenset(): 0.0, frozenset({"a"}): 0.25, frozenset({"b"}): 0.5, frozenset({"a", "b"}): 0.75}
    got = average_steps(additive, ["a", "b"])
    assert got == {"a": 0.25, "b": 0.5}, (
        f"got {got}: when the effects simply add, a name's step is the same in every order"
    )
    three = {frozenset(): 0.0, frozenset({"a"}): 1.0, frozenset({"b"}): 0.0, frozenset({"c"}): 0.0,
             frozenset({"a", "b"}): 1.0, frozenset({"a", "c"}): 1.0, frozenset({"b", "c"}): 0.0,
             frozenset({"a", "b", "c"}): 1.0}
    got = average_steps(three, ["a", "b", "c"])
    assert abs(sum(got.values()) - 1.0) < 1e-9 and abs(got["a"] - 1.0) < 1e-9, (
        f"got {got}: here only 'a' matters, so it should get the whole 1.0; average over ALL 3! "
        "orders, and the averages must add up to score[all] - score[none]"
    )
    print("exercise 5 looks right")


_try("exercise 5", _check_average_steps)

In [ ]:
ATTRIBUTION: dict[str, dict[str, list[float]]] = {}


def _all_orders() -> None:
    first = {leak: [] for leak in LEAKS}
    last = {leak: [] for leak in LEAKS}
    mean_step = {leak: [] for leak in LEAKS}
    copies_by_customer, gaps = [], []
    for i in range(N_BOOKS):
        table = {}
        for r in range(len(LEAKS) + 1):
            for present in itertools.combinations(LEAKS, r):
                table[frozenset(present)] = evaluate(i, ALL_FIXED - frozenset(present))
        steps = _call("average_steps", average_steps, table, list(LEAKS))
        everything = frozenset(LEAKS)
        gaps.append(table[everything] - table[frozenset()])
        copies_by_customer.append(table[frozenset({"duplicate", "temporal"})] - table[frozenset({"temporal"})])
        for leak in LEAKS:
            first[leak].append(table[frozenset({leak})] - table[frozenset()])
            last[leak].append(table[everything] - table[everything - {leak}])
            mean_step[leak].append(steps[leak])
    ATTRIBUTION.update(first=first, last=last, average=mean_step)
    print(f"AUC added by each leak, mean over {N_BOOKS} books (standard error in brackets)")
    print(f"  {'leak':<15}{'added to an honest protocol':>29}{'added last':>18}{'average over every order':>28}")
    for leak in LEAKS:
        print(f"  {leak:<15}{_signed(np.mean(first[leak])):>20} ({_se(first[leak]):.3f})"
              f"{_signed(np.mean(last[leak])):>10} ({_se(last[leak]):.3f})"
              f"{_signed(np.mean(mean_step[leak])):>14} ({_se(mean_step[leak]):.3f})")
    total = sum(float(np.mean(mean_step[leak])) for leak in LEAKS)
    print(f"the five averages add up to {_signed(total)}; naive minus honest is {_signed(np.mean(gaps))}"
          + (" (they agree)" if abs(total - float(np.mean(gaps))) < 1e-9 else " (they DISAGREE)"))
    spread = {leak: abs(np.mean(first[leak]) - np.mean(last[leak])) for leak in LEAKS}
    widest = max(spread, key=spread.get)
    lo, hi = sorted((float(np.mean(first[widest])), float(np.mean(last[widest]))))
    print(f"the leak whose cost depends most on the order is {widest}: {_signed(lo)} to {_signed(hi)}")
    biggest = max(LEAKS, key=lambda leak: np.mean(mean_step[leak]))
    print(f"the leak with the largest average cost here is {biggest}")
    # The duplicate leak where it can pay: the same repeats, with the folds cut by customer at random
    # (the calendar ignored) instead of forward in time. Both numbers come from the table above.
    cohort = np.array(first["duplicate"])
    by_customer = np.array(copies_by_customer)
    print("the duplicate leak, added to the honest protocol (cohort-forward folds): "
          f"{_signed(cohort.mean())} ({_se(cohort):.3f}); added to the same protocol with its folds cut by "
          f"customer at random instead (temporal present): {_signed(by_customer.mean())} ({_se(by_customer):.3f})")
    if by_customer.mean() > 2 * _se(by_customer) and cohort.mean() < by_customer.mean():
        print("Repeats cost most where the folds can put a copy and its original on different sides.")
    else:
        print("On these books the repeats do not cost clearly more under customer folds than under forward folds.")
    print("All of these belong to THIS planted book. What carries over is the method: measure each "
          "leak on your own data, in more than one order, before you decide which to fix first.")


if _DEMOS:
    _try("all orders", _all_orders, needs=("exercise 5",))

## 9. Common mistakes

* **Reporting the first AUC you get.** A cross-validated score is a claim about deployment. If you
  have not said which rows deployment will score (new customers? next month?), you have not said
  what the split must imitate.
* **Splitting by row because the data has one row per ... something.** When one customer, patient,
  machine or document yields several rows and the model will be used on new ones, the unit of the
  split is that thing, not the row. If it will instead re-score the same customers it was trained
  on, their earlier rows are fair evidence and a forward split by row is the honest one: say which
  deployment you mean, then split to match.
* **Fitting anything before the split.** Scalers, encoders, imputers, feature selection and
  resampling are all models of the data. Fit them on the training fold, inside the loop.
* **Trusting a threshold scan.** A column can be a leak and still look like a merely good feature.
  Ask when each value was written, not only how well it predicts.
* **Treating a clean audit as a clean model.** The checks look for known routes. They do not
  prove there are none, and the number they cannot see is what you will meet in production.
* **Quoting one leak's cost as a fixed number.** Leaks overlap; the cost depends on the order you
  fix them in and on what else is still broken.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## 10. Self-check

1. A model will be used to score customers it has never seen. Your cross-validation puts rows of
   the same customer in both training and test folds, and the score is high. Which statement is
   right?
   - (a) The score is fine: the model saw more data per customer, so it is better trained
   - (b) The score is optimistic if customers have stable traits the model can memorise, whether or
     not it shows in this run, so the folds should keep each customer whole
   - (c) The score is pessimistic, because the same customer appears in two places
   - (d) Nothing can be said until the test fold is made larger

2. A pipeline standardises with the mean of all rows, then splits. Its fold-isolation test reports
   that other test rows move the scores, but the AUC with and without the fix is the same to the
   third decimal. What is the sensible conclusion?
   - (a) The test is wrong, because a leak that changes nothing is not a leak
   - (b) It proves the whole protocol is clean
   - (c) The AUC must have been computed on the wrong rows
   - (d) It is a leak in the letter and harmless on these data; fix it anyway, because the same
     habit applied to a label-reading step is not harmless

3. A late-written column is dropped and the cross-validated AUC falls a lot. A teammate says "so
   the column was a strong feature; dropping it was a mistake". What is the reply?
   - (a) The AUC fell because the model can no longer see what it would not have had at scoring
     time: what fell was the inflation
   - (b) Agreed: AUC falling means the model got worse
   - (c) Cross-validation cannot compare models with different columns
   - (d) Only an accuracy metric can be compared before and after

4. Two leaks, A and B, memorise the same column. Fixing A first lowers the AUC a lot and fixing B
   afterwards lowers it a little. Fixing B first gives the reverse. Which report is honest?
   - (a) A's cost is the first number, B's the second
   - (b) B's cost is the first number, A's the second
   - (c) The cost of each depends on the order; report both orders, or an average over orders
   - (d) The smaller number is noise and can be ignored

5. Two copies of every record sit in a table, and every fold is built by taking the rows inside a
   window of time. Does the duplicate leak need fixing?
   - (a) Yes, always: duplicates always cross the folds
   - (b) No, because windows of time cannot contain duplicates
   - (c) Only if the label is imbalanced
   - (d) Only if the copies can land in different windows; copies with one timestamp share a
     window, but a split that ignores time would separate them

<details><summary>Answers</summary>

Answers come with this lesson's worked solution when you enrol on Synapsa.
</details>

In [ ]:
def _report() -> None:
    if not ATTRIBUTION:
        print("report: the all-orders table above has not run, so there is no cost to rank.")
        return
    book, folds, pipeline, used = build_protocol(BOOKS[0], frozenset(), 0)
    found = _call("diagnose", diagnose, book, folds, pipeline, used)
    ranked = sorted(LEAKS, key=lambda leak: -float(np.mean(ATTRIBUTION["average"][leak])))
    print("clinic report: the naive protocol, book 0")
    print(f"  the checks fire for {len(found)} of {len(LEAKS)} leaks: {', '.join(found) if found else 'none'}")
    print("  average AUC added, largest first: "
          + ", ".join(f"{leak} {_signed(np.mean(ATTRIBUTION['average'][leak]))}" for leak in ranked))
    print("  ranked by average cost, a first guess at which leak to look at first: " + ", ".join(ranked)
          + " (the cost of one fix also depends on what is fixed before it)")
    print("  every leak the checks find is worth fixing: a check says a leak is possible whether or not "
          "it paid on this book.")


if _DEMOS:
    _try("clinic report", _report, needs=tuple(_EXERCISES))

## What you built, and where it goes next

`split_leaks`, `feature_audit` and `fold_isolation` are the three audits a split, a column and a
pipeline each need; `diagnose` runs them together on any protocol you can describe as folds plus a
`fit_predict`; `average_steps` turns a set of ablation runs into a cost per leak that does not
depend on the order you happened to fix them in. The same three audits apply unchanged to a model
you did not write: the split and feature audits never call the model, and `fold_isolation` calls it
only through `fit_predict`; between them they look for all five leaks.

Left out on purpose: leakage through the *choice* of model or hyperparameters on the test fold
(T01-L01's peeking measurement is the place to start), near-duplicates that are not exactly equal,
and label leakage that comes from how the label was produced. Each needs a check of its own, and
none can be studied honestly until the five here are under control. Also not measured: other
models. Every size printed is for a ridge logistic regression over encoded categories on these
books; a model that can memorise individual rows would pay a different price for the same leaks.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    _ALL_CHECKS = (
        ("exercise 1", _check_split_leaks, ()),
        ("exercise 2", _check_feature_audit, ()),
        ("exercise 3", _check_fold_isolation, ()),
        ("exercise 4", _check_diagnose, ("exercise 1", "exercise 2", "exercise 3")),
        ("exercise 5", _check_average_steps, ()),
    )
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check, _needs in _ALL_CHECKS:
            _try(_name, _check, needs=_needs)
    _progress_board()
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero. Inside a notebook kernel the
    # board above has already said so, in a line rather than a traceback.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))

<!-- COMMONS NOTICE v1 · generated by tools/notebooks.py · do not edit by hand -->
---
**Synapsa Commons** · © 2026 RealAI · licensed under [CC BY-NC-SA
4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)

**You may** use this lesson to learn and to teach, and copy, fork, share and adapt it.

**You must** credit "Synapsa Commons by RealAI" with a link to
https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials,
say what you changed, and share anything you adapt under this same licence.

**You may not** use it, or anything adapted from it, in a way primarily intended for
commercial advantage or payment: for example selling it, charging for a course, bootcamp or
training built on it, or packaging it into a paid product or service. For a commercial
licence, contact [RealAI](https://www.realai.eu/contact).

Third-party material in this lesson keeps its own licence, named in `assets/SOURCE.md` or
`claims.yaml`. The Synapsa name and logo belong to RealAI and are not licensed. This summary
is not the licence: the [legal
code](https://creativecommons.org/licenses/by-nc-sa/4.0/legalcode) governs.